# 准备 CC3M 数据集和 DataLoader

在 [基于 transformers 搭建 Llava.ipynb](<基于 transformers 搭建 Llava.ipynb>) 中，我们搭建好自己的 LLaVA 模型并保存到 `show_model/model001`。目前我们的模型只有 CLIP-ViT 和 Qwen-1.5-4B 各自的预训练权重，LLaVA 模型的对齐单元（MLP）以及对 LLM 做的训练还没有进行，所以模型还并不具备读图能力。

为了训练出一个与 LLaVA 功能相似的模型，我们选择用 [LLaVA-CC3M-Pretrain-595K](https://huggingface.co/datasets/liuhaotian/LLaVA-CC3M-Pretrain-595K) 的随机抽取的一个小子集，做**小规模**的 LLaVA 第一阶段（特征对齐）复现。

训练方式：

- 视觉塔（CLIP ViT-L/14-336）冻结；
- LLM（Qwen1.5-4B-Chat）用 LoRA 训练；
- 视觉-文本投影层（`multi_modal_projector`，两层 MLP）全量训练。

In [ ]:
# pip install transformers==4.41.2 peft==0.11.1 huggingface_hub modelscope

## 与原始 LLaVA 论文的差异

### LLaVA 训练过程及数据简介

在 [CLIP 和 Llama.md](<CLIP 和 Llama.md>) 中，提到 LLaVA 的原始训练过程分为两个阶段：

- **特征对齐预训练**：冻结 ViT 和 LLM 权重，使用**图像-文本对数据集**训练投影/对齐单元。  
- **端到端微调**：冻结 ViT 权重，使用**视觉指令数据集**同时调整对齐单元和 LLM 的权重。

具体而言，图像-文本对数据集就是我们将要使用的经过 LLaVA 团队筛选的 CC3M 数据集，视觉指令数据集则是 LLaVA 团队在 COCO 2017 数据集的基础上自行构建的。**前一个数据集中的指令语义非常单一，基本停留在图像内容描述层面**（如“这张图片里有什么？”）。后一个数据集才涉及图片具体的提问，甚至基于图片进行推理。

### 我们将要进行的训练过程

我们的训练过程只有一个阶段：

- **同时进行特征对齐 + LLM 的轻量视觉适配**：冻结 ViT 权重，使用**图像-文本对数据集**训练对齐单元和 LLM，其中 LLM 使用 LoRA 进行高效微调。

这种设计会带来一些需要留意的影响：

- **训练初期可能不如原版稳定**：Connector 和 LLM LoRA 同时从早期开始联合优化，缺少了原版 Stage 1 中 LLM 冻结提供的“安全环境”，损失曲线可能更易波动。
- **指令遵循能力有限**：由于训练数据是图文对而非指令数据，模型学到的主要是“用语言描述视觉特征”，而非原版 Stage 2 后的复杂指令跟随、多轮对话或基于图像的推理。它具备的是**基础的看图说话能力**，而不是完整的视觉指令遵循。
- **LoRA 的正则化效应提供了补偿**：相比全量微调 LLM，LoRA 只更新少量参数，降低了模型在缺乏独立对齐阶段时“跑偏”或遗忘预训练知识的风险，使这种单阶段联合训练在实践中相对可控。
- **性价比优势明显**：训练成本显著低于完整两阶段流程，适合在有限算力下快速得到一个基础可用的模型。

本仓库的目标是**在有限算力预算下，快速得到一个效果可接受、具备基本视觉指令跟随能力的模型**。完全模仿原论文进行训练成本较高，因此选择了这样的训练方式作为**更具性价比的起点**。如果后续需要更完整的指令遵循能力，可以在此基础上引入 LLaVA-Instruct-150K 和 COCO 2017 图片，补充第二阶段的指令微调。

## 下载数据集

魔塔社区并没有提供 LLaVA 的 CC3M 数据集。我们需要使用镜像站或 HuggingFace 主站进行数据集下载。这里默认在 AutoDL 上使用学术加速，从 HuggingFace 主站 下载数据集。以下命令需要安装好 `huggingface_hub`。

```bash
unset HF_ENDPOINT
source /etc/network_turbo
hf download liuhaotian/LLaVA-CC3M-Pretrain-595K --local-dir /root/autodl-tmp/CC3M
```

数据集默认下载到数据盘中，大小约 7GB。图片文件是在压缩包中的，后续需要根据需要解压。

### 数据格式

`chat.json` 的每条记录形如：

```json
{
  "conversations": [
    {"from": "human", "value": "Provide a brief description of the given image.\n<image>"},
    {"from": "gpt", "value": "olive oil is a healthy ingredient used liberally ."}
  ],
  "image": "GCC_train_002582585.jpg",
  "id": "GCC_train_002582585"
}
```

- `<image>` 已经包含在 human 的文本里，位置可能在开头或结尾；
- 图片名是扁平的 `GCC_train_*.jpg`，都在 `images.zip` 的根目录；
- 我们只需要把选中的图片从 zip 里取出来，不需要解压全部 59 万张。

### 检查数据

运行下方的代码块，对下载好的数据做数量上和读取功能上的检查。为了后续校验数据加载器能否被我们的模型使用，我们这里同时检查是否运行过 [基于 transformers 搭建 Llava.ipynb](<基于 transformers 搭建 Llava.ipynb>) 将初始 LLaVA 模型保存到指定路径。

In [ ]:
from pathlib import Path

MODEL_DIR = "show_model/model001"   # 搭建 notebook 保存的模型目录
DATA_ROOT = Path("/root/autodl-tmp/CC3M")

CHAT_PATH = DATA_ROOT / "chat.json"
IMAGE_ZIP = DATA_ROOT / "images.zip"

SUBSET_SIZE = 5000   # 训练样本数
VAL_SIZE = 100       # 验证样本数
SEED = 42

IMAGE_DIR = DATA_ROOT / "images"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)

assert Path(MODEL_DIR).exists(), f"{MODEL_DIR} 不存在，请先运行搭建 notebook"
assert CHAT_PATH.exists(), f"{CHAT_PATH} 不存在"
assert IMAGE_ZIP.exists(), f"{IMAGE_ZIP} 不存在"

print("data root:", DATA_ROOT)
print("model dir:", Path(MODEL_DIR).resolve())

## 划分和解压

### 将数据集划分为训练集和验证集

在 `chat.json` 中随机抽出上面代码块指定的样本数量的样本（这里还没有加载图片，只是把图片对应的文本信息加载进来，后续直接通过 json 中保存的文件名解压对应的图片），加载进内存。

In [ ]:
import json
import random

chat_path = CHAT_PATH
print(chat_path)

with open(chat_path, encoding="utf-8") as f:
    all_records = json.load(f)

print("total records:", len(all_records))

rng = random.Random(SEED)
indices = rng.sample(range(len(all_records)), SUBSET_SIZE + VAL_SIZE)
subset = [all_records[i] for i in indices]
del all_records

# 保证每条样本的 human 文本里有且只有一个 <image>
for rec in subset:
    human = rec["conversations"][0]["value"]
    if human.count("<image>") != 1:
        rec["conversations"][0]["value"] = "<image>\n" + human.replace("<image>", "").strip()

train_records = subset[:SUBSET_SIZE]
val_records = subset[SUBSET_SIZE:]

with open(DATA_ROOT / "chat_train.json", "w", encoding="utf-8") as f:
    json.dump(train_records, f, ensure_ascii=False)
with open(DATA_ROOT / "chat_val.json", "w", encoding="utf-8") as f:
    json.dump(val_records, f, ensure_ascii=False)

print("train:", len(train_records), "val:", len(val_records))

### 解压图片

`images.zip` 里是 59 万多张扁平的 `GCC_train_*.jpg`；这里只解压选中的 5100 张到 `images/`，避免生成 59 万个文件。

In [ ]:
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

selected_names = [rec["image"] for rec in subset]
todo = [name for name in selected_names if not (IMAGE_DIR / name).exists()]
print("images to extract:", len(todo))

with zipfile.ZipFile(IMAGE_ZIP) as z:
    # zip 内部是扁平结构，这里用 basename 做一层兼容
    available = {
        Path(info.filename).name: info.filename
        for info in z.infolist()
        if not info.is_dir()
    }
    missing = [name for name in todo if name not in available]
    print("missing in zip:", len(missing), missing[:3])

    for name in tqdm(todo):
        if name not in available:
            continue
        (IMAGE_DIR / name).write_bytes(z.read(available[name]))

检查一下图文信息是否已经完整，训练集测试集的样本数量是否符合设置。

In [ ]:
missing_images = [
    rec["image"] for rec in subset if not (IMAGE_DIR / rec["image"]).exists()
]
print("missing images:", len(missing_images), missing_images[:5])

if missing_images:
    missing_set = set(missing_images)
    train_records = [rec for rec in train_records if rec["image"] not in missing_set]
    val_records = [rec for rec in val_records if rec["image"] not in missing_set]
    with open(DATA_ROOT / "chat_train.json", "w", encoding="utf-8") as f:
        json.dump(train_records, f, ensure_ascii=False)
    with open(DATA_ROOT / "chat_val.json", "w", encoding="utf-8") as f:
        json.dump(val_records, f, ensure_ascii=False)

print("usable train:", len(train_records), "val:", len(val_records))
print("extracted images:", len(list(IMAGE_DIR.glob("*.jpg"))))

## 创建 torch.Dataset